In [ ]:
# LOAD EMBEDDING
import pickle

EMBEDDING_FILE = "moho_embeddings_images.pkl"

with open(EMBEDDING_FILE, "rb") as f:
    data = pickle.load(f)

print("✅ Loaded:", len(data))

# lấy dimension
dim = len(data[0]["vector"])
print("Vector dim:", dim)

✅ Loaded: 949
Vector dim: 512


In [13]:
from pymilvus import MilvusClient

# Kết nối tới Milvus server
milvus_client = MilvusClient(uri="http://localhost:19530")

# Kiểm tra kết nối bằng cách liệt kê collection
try:
    collections = milvus_client.list_collections()
    print("Kết nối thành công! Danh sách collection:", collections)
except Exception as e:
    print("Kết nối thất bại:", str(e))

Kết nối thành công! Danh sách collection: ['ThucTap_image', 'my_rag_collection', 'image_collection']


In [14]:
collection_name = "ThucTap_image"

In [43]:
# ==============================
# 2 RESET HARD
# ==============================

collection_name = "ThucTap_image"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)
    print("🗑️ Đã xóa collection")

print("Collections hiện tại:", milvus_client.list_collections())

Collections hiện tại: ['my_rag_collection', 'image_collection']


In [ ]:
# ==============================
# 3. CREATE COLLECTION
# ==============================

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=dim,
    metric_type="COSINE",
    enable_dynamic_field=True
)

print("✅ Collection created")

✅ Collection created


In [15]:
print(milvus_client.list_collections())

['ThucTap_image', 'my_rag_collection', 'image_collection']


In [16]:
print(milvus_client.list_indexes("ThucTap_image"))

['vector']


In [17]:
milvus_client.describe_index(
    collection_name="ThucTap_image",
    index_name="vector"
)

{'M': '16',
 'efConstruction': '200',
 'metric_type': 'COSINE',
 'index_type': 'HNSW',
 'field_name': 'vector',
 'index_name': 'vector',
 'total_rows': 0,
 'indexed_rows': 0,
 'pending_index_rows': 0,
 'state': 'Finished'}

In [ ]:
# ==============================
# 4. PREPARE DATA
# ==============================

milvus_data = []

for i, item in enumerate(data):
    milvus_data.append({
        "id": i,  # unique cho mỗi ảnh
        "vector": item["vector"].tolist(),
        "product_id": int(item["product_id"])
    })

print("Ready insert:", len(milvus_data))

Ready insert: 949


In [49]:
# ==============================
# 5. INSERT DATA
# ==============================

milvus_client.insert(
    collection_name=collection_name,
    data=milvus_data
)

print("✅ Insert done")

✅ Insert done


In [50]:
# ==============================
# 8. LOAD COLLECTION
# ==============================

milvus_client.load_collection(collection_name)

print("✅ Loaded into memory")

✅ Loaded into memory


# test search

In [56]:
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient

# ==============================
# 1. LOAD MODEL
# ==============================
model = SentenceTransformer("clip-ViT-B-32-multilingual-v1")

# ==============================
# 2. CONNECT MILVUS
# ==============================
milvus_client = MilvusClient(uri="http://localhost:19530")

collection_name = "ThucTap_image"



c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The tokenizer you are loading from 'clip-ViT-B-32-multilingual-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [59]:
# ==============================
# 3. INPUT TEXT (VIETNAMESE)
# ==============================
query = "tủ quần áo có cửa bằng kính"

# encode text -> vector
query_vector = model.encode(query).tolist()

# ==============================
# 4. SEARCH
# ==============================
results = milvus_client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=5,
    search_params={"metric_type": "COSINE"},
    output_fields=["product_id"]   # 🔥 BẮT BUỘC
)

# ==============================
# 5. PRINT RESULT
# ==============================
for i, res in enumerate(results[0]):
    print(f"\nTop {i+1}")
    print("Score:", res["distance"])
    print("Product ID:", res["entity"]["product_id"])


Top 1
Score: 0.2938145399093628
Product ID: 143

Top 2
Score: 0.28568801283836365
Product ID: 175

Top 3
Score: 0.2854679524898529
Product ID: 153

Top 4
Score: 0.2843511700630188
Product ID: 179

Top 5
Score: 0.2838105857372284
Product ID: 180


In [1]:
from sentence_transformers import SentenceTransformer
from PIL import Image
from pymilvus import MilvusClient

# ==============================
# 1. LOAD MODEL
# ==============================
model = SentenceTransformer("clip-ViT-B-32")

# ==============================
# 2. CONNECT MILVUS
# ==============================
milvus_client = MilvusClient(uri="http://localhost:19530")
collection_name = "ThucTap_image"

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# ==============================
# 3. LOAD IMAGE QUERY
# ==============================
image_path = r"D:\Downloads\bo-ban-an-tron-sang-trong (avt).jpg"   # 👉 đổi thành ảnh của bạn
image = Image.open(image_path)

# ==============================
# 4. ENCODE IMAGE → VECTOR
# ==============================
query_vector = model.encode(image).tolist()

# ==============================
# 5. SEARCH
# ==============================
results = milvus_client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=5,
    search_params={"metric_type": "COSINE"}, # sử dụng độ đo Cosine, Milvus mặc định hiểu rằng: Giá trị càng cao = Độ tương đồng càng lớn. Do đó, khi trả về kết quả cho hàm search, Milvus luôn tự động sắp xếp danh sách theo thứ tự ưu tiên từ cao xuống thấp.
    output_fields=["product_id"]   # 🔥 nhớ cái này
)

# ==============================
# 6. PRINT RESULT
# ==============================
for i, res in enumerate(results[0]):
    print(f"\nTop {i+1}")
    print("Score:", res["distance"])
    print("Product ID:", res["entity"]["product_id"])


Top 1
Score: 0.8011909127235413
Product ID: 23

Top 2
Score: 0.7994109988212585
Product ID: 5

Top 3
Score: 0.7945134043693542
Product ID: 87

Top 4
Score: 0.7940649390220642
Product ID: 19

Top 5
Score: 0.7938787341117859
Product ID: 5


Model này thực chất:

🖼 Image encoder = giống clip-ViT-B-32 (OpenAI)

📝 Text encoder = đã fine-tune đa ngôn ngữ

# check thời gian tìm, chỉnh index

In [18]:
import time
import pickle
import numpy as np
from tqdm import tqdm
from pymilvus import MilvusClient

# --- 1. KHAI BÁO & KẾT NỐI ---
EMBEDDING_FILE = "moho_embeddings_images.pkl"
COLLECTION_NAME = "ThucTap_image"

milvus_client = MilvusClient(uri="http://localhost:19530")

# --- 2. CHUẨN BỊ BỘ TRUY VẤN ---
with open(EMBEDDING_FILE, "rb") as f:
    query_source = pickle.load(f)

print(f"✅ Đã nạp {len(query_source)} vector mẫu để đo hiệu năng.")

# --- 3. HÀM ĐO LATENCY ---
def run_latency_test(index_label):
    # Load collection vào RAM
    milvus_client.load_collection(COLLECTION_NAME)
    
    n_queries = len(query_source)
    latencies = []

    print(f"\n🚀 Đang đo hiệu năng hệ thống: [{index_label}]")

    for i in tqdm(range(n_queries)):
        v_query = query_source[i]['vector']
        
        # --- ĐO THỜI GIAN SEARCH ---
        start = time.perf_counter()
        milvus_client.search(
            collection_name=COLLECTION_NAME,
            data=[v_query],
            limit=10 # Giả định tìm Top 10
        )
        end = time.perf_counter()
        
        # Chuyển sang miliseconds
        latencies.append((end - start) * 1000)

    # --- TÍNH TOÁN THỐNG KÊ ---
    avg_lat = np.mean(latencies)
    p95_lat = np.percentile(latencies, 95)
    p99_lat = np.percentile(latencies, 99)

    # --- XUẤT BÁO CÁO ---
    print("\n" + "="*60)
    print(f"📊 BÁO CÁO HIỆU NĂNG: {index_label}")
    print("-" * 60)
    print(f"⏱️ Average Latency : {avg_lat:.2f} ms")
    print(f"⏱️ P95 Latency     : {p95_lat:.2f} ms")
    print(f"⏱️ P99 Latency     : {p99_lat:.2f} ms")
    print("="*60 + "\n")

# --- 4. THỰC THI ---

# 4.1 Đo hiệu năng với Index hiện tại (Lúc này đang là AUTOINDEX)
# Hàm này sẽ load collection vào RAM và thực hiện search thử 949 mẫu để tính thời gian trung bình.
run_latency_test("AUTOINDEX")

# 4.2 Bắt đầu quá trình chuyển đổi cấu hình Index sang HNSW
print("⚙️ Đang cấu hình lại Index sang HNSW...")

# Bước 1: Giải phóng Collection khỏi RAM. 
# Milvus không cho phép thay đổi Index khi dữ liệu đang được load trên RAM.
milvus_client.release_collection(COLLECTION_NAME)

# Bước 2: Xóa Index cũ. 
# Cần xóa index tên là "vector" (tên mặc định của trường chứa embedding) trước khi tạo cái mới.
milvus_client.drop_index(COLLECTION_NAME, "vector")

# Bước 3: Khởi tạo bộ tham số mới cho Index
index_params = milvus_client.prepare_index_params()

# Bước 4: Định nghĩa cấu trúc HNSW
index_params.add_index(
    field_name="vector",        # Tên trường vector trong collection
    index_type="HNSW",          # Loại Index: Hierarchical Navigable Small World (Đồ thị phân cấp)
    metric_type="COSINE",       # Phương pháp đo khoảng cách: Cosine Similarity (phù hợp cho ảnh/văn bản)
    params={
        "M": 16,                # Số lượng liên kết tối đa cho mỗi node trong đồ thị. 
                                # (Thường từ 4-64. M càng cao: độ chính xác tăng nhưng tốn RAM hơn)
        
        "efConstruction": 200   # Phạm vi tìm kiếm khi đang xây dựng đồ thị Index.
                                # (Giá trị càng cao: Index càng chất lượng nhưng thời gian tạo Index lâu hơn)
    }
)

# Bước 5: Ra lệnh cho Milvus bắt đầu xây dựng Index mới dựa trên tham số đã khai báo
milvus_client.create_index(COLLECTION_NAME, index_params)

# Bước 6: Chạy lại bài test để so sánh kết quả sau khi đã tối ưu
# Lúc này hàm run_latency_test sẽ load Index HNSW mới vào RAM và đo lại tốc độ.
run_latency_test("OPTIMIZED_HNSW")

✅ Đã nạp 949 vector mẫu để đo hiệu năng.

🚀 Đang đo hiệu năng hệ thống: [AUTOINDEX]


100%|██████████| 949/949 [00:08<00:00, 106.26it/s]



📊 BÁO CÁO HIỆU NĂNG: AUTOINDEX
------------------------------------------------------------
⏱️ Average Latency : 9.23 ms
⏱️ P95 Latency     : 12.84 ms
⏱️ P99 Latency     : 17.62 ms

⚙️ Đang cấu hình lại Index sang HNSW...

🚀 Đang đo hiệu năng hệ thống: [OPTIMIZED_HNSW]


100%|██████████| 949/949 [00:09<00:00, 97.14it/s] 


📊 BÁO CÁO HIỆU NĂNG: OPTIMIZED_HNSW
------------------------------------------------------------
⏱️ Average Latency : 10.16 ms
⏱️ P95 Latency     : 11.20 ms
⏱️ P99 Latency     : 15.23 ms

